# Plant Disease — Model B: field-ready classifier (38 PlantVillage classes)  

**Backbone:** EfficientNetV2-B0 · **Input:** 256×256 · **Output:** 38 classes  
**Deploy file:** `model_b_combined.keras` + `class_names.json`

Lab-only baseline (old Model A) is in a separate notebook: `train_model_a_kaggle.ipynb`.

Phase checkpoints (`best_model_b_*.keras`) are training artifacts only — do not deploy them.

### Data sources
| Source | Role |
|--------|------|
| PlantVillage | Lab baseline — all 38 classes |
| PlantDoc train | Real-field images — robustness |
| PlantCity (Kaggle Input) | Extra multi-crop field boost |
| Tomato Disease Multiple Sources (Kaggle Input) | Tomato-focused lab/field diversity |

**Benchmark:** PlantDoc test (never trained on, MD5-deduplicated)

### Training schedule
| Phase | Epochs | Steps/epoch | Field mix | Mixup |
|-------|:------:|:-----------:|:---------:|:-----:|
| Warmup (frozen backbone) | 4 | 500 | 50% | No |
| Finetune (full model) | 14 | 1000 | 65% | Yes (α=0.2) |
| Polish (low LR) | 6 | 600 | 85% | No |

Hard-class boosting (3× sampling weight) for known weak classes.  
EMA (Polyak averaging) enabled — acts as a free ensemble.  
TTA: 5 views (identity + LR flip + UD flip + 85% crop + 90% crop).

### Deploy contract (Flask must match exactly)
- **Input:** float32 RGB, shape `(1, 256, 256, 3)`, values **0…255** (NOT /255)  
  Preprocessing = center square crop → resize 256 (see `serve_image` below)  
- **Output:** softmax over `class_names.json` (38 classes)  
- **No `preprocess_input`** — EfficientNetV2 handles its own internal normalisation  


In [ ]:
# ============================================================================
# Tomato leaf disease classifier: PlantVillage (lab) + PlantDoc (field)  - v2
# ============================================================================
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '1')   # hide TF C++ INFO spam, keep warnings/errors

import re, glob, json, time, shutil, hashlib, zipfile, subprocess
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from PIL import Image
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support

# ----------------------------------------------------------------- config ---
WORK = '/kaggle/working'
INPUT = '/kaggle/input'
SEED = 42
CROP = 'Tomato'                     # None -> all 38 PlantVillage classes

# Folder that directly contains train/ + valid/ (PV) or train/ + test/ (PlantDoc). None = auto-find.
PV_ROOT = None
PD_ROOT = None
EXTRA_ROOT = None                   # tomato-disease-multiple-sources (folder containing train/)

BACKBONE = 'efficientnetv2s'        # efficientnetv2b0 | efficientnetv2b2 | efficientnetv2s | convnext_tiny
IMG_SIZE = 288                      # try 352 if time allows (bacterial vs septoria spots are tiny)
BATCH = 32                          # 16 if OOM
MIXED_PRECISION = True

LABEL_SMOOTHING = 0.1
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 1.0
CUTMIX_PROB = 0.5
DROPOUT = 0.3
WEIGHT_DECAY = 0.05
EMA_DECAY = 0.998
EMA_EVERY = 8

FIELD_BALANCE_TEMP = 0.5            # 0 = classes equally likely, 1 = natural counts
MIN_FIELD_PER_CLASS = 10            # rarer field classes join the lab pool (no extreme oversampling)
PLANTDOC_SHARE = 0.5                # share of field samples from PlantDoc-train (rest: extra wild photos)
EXTRA_WILD_TO_FIELD = True          # non-PlantVillage photos of the multi-source set feed the field branch
EXTRA_DROP_ROBOFLOW = True          # '*.rf.*' files are Roboflow re-exports (PlantDoc is on Roboflow)
EXTRA_CAP = 1500                    # per class
MAX_PER_SOURCE_FOLDER = 1500
PD_VAL_FRACTION = 0.15
NEAR_DUP_BITS = 10                  # dHash distance (of 64) treated as "same photo"
GATE_TARGET_ACC = 0.85
EXTRA_FIELD_DIRS = []               # class-folder roots of your own field photos

PHASES = [
    dict(name='warmup',   epochs=3,  steps=300, lr=1e-3, field_mix=0.40, mix=False,
         finetune=False, ema=False, patience=3),
    dict(name='finetune', epochs=15, steps=400, lr=1e-4, field_mix=0.50, mix=True,
         finetune=True,  ema=True,  patience=5),
    dict(name='polish',   epochs=5,  steps=300, lr=2e-5, field_mix=0.60, mix=False,
         finetune=True,  ema=True,  patience=3),
]

MODEL_PATH = f'{WORK}/model_tomato.keras'
CKPT_PATH = f'{WORK}/checkpoint.weights.h5'
FP_CACHE_NAME = 'fingerprints_cache.json'
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')
AUTOTUNE = tf.data.AUTOTUNE

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)
keras.utils.set_random_seed(SEED)


def setup_precision():
    gpus = tf.config.list_physical_devices('GPU')
    if MIXED_PRECISION and gpus:
        cc = tf.config.experimental.get_device_details(gpus[0]).get('compute_capability', (0, 0))
        if cc and cc[0] >= 7:
            keras.mixed_precision.set_global_policy('mixed_float16')
            return 'mixed_float16'
    return 'float32'


PRECISION = setup_precision()

BACKBONES = {  # all expect raw 0..255 input (they normalise internally)
    'efficientnetv2b0': keras.applications.EfficientNetV2B0,
    'efficientnetv2b2': keras.applications.EfficientNetV2B2,
    'efficientnetv2s': keras.applications.EfficientNetV2S,
    'convnext_tiny': keras.applications.ConvNeXtTiny,
}
assert BACKBONE in BACKBONES, f'unknown backbone {BACKBONE}'


# ------------------------------------------------------------------ utils ---
def sh(cmd):
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)


def disk_free(label=''):
    print(f'Disk [{label}]: {shutil.disk_usage(WORK).free / 1e9:.1f} GB free')


def stable_bucket(key, mod=100):
    return int(hashlib.md5(key.encode()).hexdigest()[:8], 16) % mod


def file_md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def image_files(folder):
    try:
        return sorted(f for f in os.listdir(folder) if f.lower().endswith(IMG_EXTS))
    except OSError:
        return []


def subdirs(folder):
    return sorted(d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d)))


NO_DESCEND = {'train', 'valid', 'val', 'test', '.git', '__MACOSX'}


def find_split_root(bases, splits, marker, max_depth=8):
    """Breadth-first search for a folder whose children include all `splits` and whose
    splits[0] contains `marker`. Never enters split folders, so image folders are never listed."""
    for base in bases:
        queue = [(base, 0)] if os.path.isdir(base) else []
        while queue:
            root, depth = queue.pop(0)
            try:
                dirs = sorted(e.name for e in os.scandir(root) if e.is_dir())
            except OSError:
                continue
            if all(s in dirs for s in splits) and os.path.isdir(os.path.join(root, splits[0], marker)):
                return root
            if depth < max_depth:
                queue += [(os.path.join(root, d), depth + 1) for d in dirs if d not in NO_DESCEND]
    return None


# ----------------------------------------------- duplicate / leak control ---
_FP = {}
_POPCOUNT = np.array([bin(i).count('1') for i in range(256)], np.uint8)


def _gray(path, size):
    with Image.open(path) as im:
        im.draft('L', (64, 64))
        return np.asarray(im.convert('L').resize(size, Image.BILINEAR), dtype=np.int16)


def _dhash(a):
    return np.packbits(a[:, 1:] > a[:, :-1]).view(np.uint64)[0]


def _fp_key(path):
    try:
        return f'{path}|{os.path.getsize(path)}'
    except OSError:
        return None


def fingerprint(path):
    """(md5, 64-bit dHash), cached in memory and in fingerprints_cache.json."""
    key = _fp_key(path)
    if key is None:
        return None, None
    if key not in _FP:
        try:
            _FP[key] = (file_md5(path), _dhash(_gray(path, (9, 8))))
        except Exception:
            _FP[key] = (None, None)
    return _FP[key]


def load_fp_cache():
    files = [os.path.join(WORK, FP_CACHE_NAME)]
    files += [p for d in range(1, 6) for p in glob.glob(os.path.join(INPUT, *['*'] * d, FP_CACHE_NAME))]
    for f in files:
        if not os.path.isfile(f):
            continue
        try:
            with open(f) as fh:
                for k, (m, d) in json.load(fh).items():
                    _FP.setdefault(k, (m, np.uint64(int(d))))
            print(f'  fingerprint cache loaded: {f}')
        except (OSError, ValueError, TypeError):
            pass


def save_fp_cache():
    data = {k: [m, str(int(d))] for k, (m, d) in _FP.items() if m is not None}
    with open(os.path.join(WORK, FP_CACHE_NAME), 'w') as fh:
        json.dump(data, fh)


def dihedral_hashes(path):
    """dHash of all 8 flips/rotations - reference side only, so flipped/rotated copies match."""
    try:
        a, b = _gray(path, (9, 8)), _gray(path, (8, 9))
    except Exception:
        return []
    r = np.rot90(b)
    return [_dhash(v) for v in (a, a[:, ::-1], a[::-1], a[::-1, ::-1],
                                r, r[:, ::-1], r[::-1], r[::-1, ::-1])]


def reference(paths):
    md5s, hashes = set(), []
    for p in paths:
        m, _ = fingerprint(p)
        if m:
            md5s.add(m)
        hashes += dihedral_hashes(p)
    return md5s, np.array(hashes, dtype=np.uint64)


def merge_refs(*refs):
    return set().union(*[r[0] for r in refs]), np.concatenate([r[1] for r in refs])


def near_dup_mask(hashes, ref, max_bits=NEAR_DUP_BITS, chunk=1024):
    hashes, ref = np.asarray(hashes, np.uint64), np.asarray(ref, np.uint64)
    out = np.zeros(len(hashes), bool)
    if not len(hashes) or not len(ref):
        return out
    for s in range(0, len(hashes), chunk):
        x = np.ascontiguousarray(np.bitwise_xor(hashes[s:s + chunk, None], ref[None, :]))
        dist = _POPCOUNT[x.view(np.uint8)].reshape(len(x), len(ref), 8).sum(-1, dtype=np.int16)
        out[s:s + chunk] = (dist <= max_bits).any(1)
    return out


removed_rows = []


def drop_matches(items, ref, tag):
    """Remove unreadable files and exact / near duplicates of the reference set."""
    if not items:
        return items
    ref_md5, ref_dh = ref
    fps = [fingerprint(p) for p, _ in items]
    readable = np.array([m is not None for m, _ in fps])
    exact = np.array([m in ref_md5 for m, _ in fps])
    near = near_dup_mask([d if d is not None else 0 for _, d in fps], ref_dh) & readable
    keep = readable & ~exact & ~near
    for (p, _), r, e, k in zip(items, readable, exact, keep):
        if not k:
            removed_rows.append({'source': tag, 'path': p,
                                 'reason': 'unreadable' if not r else 'exact' if e else 'near-duplicate'})
    print(f'  {tag}: removed {int((~keep).sum())} (exact {int(exact.sum())}, '
          f'near {int((near & ~exact).sum())}, unreadable {int((~readable).sum())})')
    return [it for it, k in zip(items, keep) if k]


# ------------------------------------------------- label harmonisation -----
FILLER = {'leaf', 'leaves', 'plant', 'plants', 'image', 'images', 'photo', 'photos',
          'disease', 'diseased', 'dataset', 'class', 'train', 'folder'}
PLANT_STOP = {'including', 'sour'}
TOKEN_SYNONYMS = {'normal': 'healthy', 'fresh': 'healthy', 'soyabean': 'soybean',
                  'soya': 'soybean', 'maize': 'corn', 'grey': 'gray', 'mould': 'mold'}


def _key(text):
    text = text.replace('___', ' ').replace('+', ' ').replace(',', ' ')
    words = [TOKEN_SYNONYMS.get(w, w) for w in re.sub(r'[^a-z0-9]+', ' ', text.lower()).split()]
    kept = [w for w in words if w not in FILLER]
    return ' '.join(kept or words)


PLANTDOC_ALIASES = {
    'Apple Scab Leaf': 'Apple___Apple_scab', 'Apple leaf': 'Apple___healthy',
    'Apple rust leaf': 'Apple___Cedar_apple_rust', 'Bell_pepper leaf': 'Pepper,_bell___healthy',
    'Bell_pepper leaf spot': 'Pepper,_bell___Bacterial_spot', 'Blueberry leaf': 'Blueberry___healthy',
    'Cherry leaf': 'Cherry_(including_sour)___healthy',
    'Corn Gray leaf spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn leaf blight': 'Corn_(maize)___Northern_Leaf_Blight',
    'Corn rust leaf': 'Corn_(maize)___Common_rust_', 'Corn leaf': 'Corn_(maize)___healthy',
    'Peach leaf': 'Peach___healthy', 'Potato leaf early blight': 'Potato___Early_blight',
    'Potato leaf late blight': 'Potato___Late_blight', 'Potato leaf': 'Potato___healthy',
    'Raspberry leaf': 'Raspberry___healthy', 'Soyabean leaf': 'Soybean___healthy',
    'Squash Powdery mildew leaf': 'Squash___Powdery_mildew', 'Strawberry leaf': 'Strawberry___healthy',
    'Tomato Early blight leaf': 'Tomato___Early_blight',
    'Tomato Septoria leaf spot': 'Tomato___Septoria_leaf_spot', 'Tomato leaf': 'Tomato___healthy',
    'Tomato leaf bacterial spot': 'Tomato___Bacterial_spot',
    'Tomato leaf late blight': 'Tomato___Late_blight',
    'Tomato leaf mosaic virus': 'Tomato___Tomato_mosaic_virus',
    'Tomato leaf yellow virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato mold leaf': 'Tomato___Leaf_Mold',
    'Tomato two spotted spider mites leaf': 'Tomato___Spider_mites Two-spotted_spider_mite',
    'grape leaf': 'Grape___healthy', 'grape leaf black rot': 'Grape___Black_rot',
}
TOMATO_ALIASES = {
    'Bacterial_spot': 'Tomato___Bacterial_spot', 'Early_blight': 'Tomato___Early_blight',
    'Late_blight': 'Tomato___Late_blight', 'Leaf_Mold': 'Tomato___Leaf_Mold',
    'Septoria_leaf_spot': 'Tomato___Septoria_leaf_spot',
    'Spider_mites Two-spotted_spider_mite': 'Tomato___Spider_mites Two-spotted_spider_mite',
    'Target_Spot': 'Tomato___Target_Spot',
    'Tomato_Yellow_Leaf_Curl_Virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato_mosaic_virus': 'Tomato___Tomato_mosaic_virus', 'healthy': 'Tomato___healthy',
}
GLOBAL_ALIASES = {
    'corn gray spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'corn cercospora spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'grape esca': 'Grape___Esca_(Black_Measles)',
    'grape black measles': 'Grape___Esca_(Black_Measles)',
    'orange huanglongbing': 'Orange___Haunglongbing_(Citrus_greening)',
    'citrus greening': 'Orange___Haunglongbing_(Citrus_greening)',
    'tomato spider mite': 'Tomato___Spider_mites Two-spotted_spider_mite',
    'tomato spider mites': 'Tomato___Spider_mites Two-spotted_spider_mite',
    'tomato two spotted spider mite': 'Tomato___Spider_mites Two-spotted_spider_mite',
    'tomato yellow curl virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'tomato ylcv': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
}


def build_mapping(root, class_names, aliases=None, tag=''):
    """folder -> class. Alias table first, then crop-anchored token match. Audited to CSV."""
    known = set(class_names)
    table = {**GLOBAL_ALIASES, **{_key(k): v for k, v in (aliases or {}).items()}}
    table = {k: v for k, v in table.items() if v in known}
    pv_tokens = {}
    for c in class_names:
        plant, disease = c.split('___', 1)
        pv_tokens[c] = (set(_key(plant).split()) - PLANT_STOP, set(_key(disease).split()))

    mapping, rows = {}, []
    for folder in subdirs(root):
        key = _key(folder)
        tokens = set(key.split())
        hit, how = table.get(key), 'alias'
        if hit is None:
            best, best_score, tie = None, 0, False
            for pv, (p, d) in pv_tokens.items():
                if not d or not (p & tokens):
                    continue
                overlap, rest = len(d & tokens), tokens - p
                if overlap == 0 or not (overlap >= len(d) - 1 or (rest and rest <= d)):
                    continue
                score = len(p & tokens) + overlap
                if score > best_score:
                    best, best_score, tie = pv, score, False
                elif score == best_score:
                    tie = True
            hit = None if tie else best
            how = 'ambiguous' if tie else (f'tokens({best_score})' if best else 'UNMAPPED')
        if hit:
            mapping[folder] = hit
        rows.append({'folder': folder, 'key': key, 'mapped_to': hit or '', 'how': how})
    pd.DataFrame(rows).to_csv(f'{WORK}/mapping_{tag}.csv', index=False)
    print(f'  {tag}: mapped {len(mapping)}/{len(rows)} folders')
    return mapping


def collect(root, mapping, cap=MAX_PER_SOURCE_FOLDER, tag=''):
    items = []
    for folder, cls in sorted(mapping.items()):
        files = image_files(os.path.join(root, folder))
        if cap and len(files) > cap:
            files = sorted(files, key=lambda f: stable_bucket(folder + f, 10 ** 9))[:cap]
        items += [(os.path.join(root, folder, f), CLASS_INDEX[cls]) for f in files]
    print(f'  {tag}: {len(items)} images')
    return items


def collect_extra(root):
    """Split the multi-source tomato set into PlantVillage copies (uuid___SOURCE names, skipped -
    already in PV) and other photos (web / Taiwan, much closer to field conditions)."""
    pv_copies, roboflow, wild = 0, 0, defaultdict(list)
    for split in ('train', 'valid'):
        d = os.path.join(root, split)
        if not os.path.isdir(d):
            continue
        for folder, cls in sorted(build_mapping(d, class_names, TOMATO_ALIASES, f'tomato-extra-{split}').items()):
            for f in image_files(os.path.join(d, folder)):
                if '___' in f:
                    pv_copies += 1
                elif EXTRA_DROP_ROBOFLOW and '.rf.' in f:
                    roboflow += 1
                else:
                    wild[CLASS_INDEX[cls]].append((os.path.join(d, folder, f), CLASS_INDEX[cls]))
    items = []
    for idx, its in sorted(wild.items()):
        items += sorted(its, key=lambda it: stable_bucket(it[0], 10 ** 9))[:EXTRA_CAP]
    print(f'  tomato-extra: {len(items)} wild photos kept | skipped {pv_copies} PlantVillage copies, '
          f'{roboflow} Roboflow re-exports')
    return items


def by_class(items):
    groups = defaultdict(list)
    for p, i in items:
        groups[i].append(p)
    return groups


def split_small(items, tag):
    groups = by_class(items)
    keep = {i: ps for i, ps in groups.items() if len(ps) >= MIN_FIELD_PER_CLASS}
    small = [(p, i) for i, ps in groups.items() if len(ps) < MIN_FIELD_PER_CLASS for p in ps]
    if small:
        print(f'  {tag}: {len(small)} images of tiny classes moved to the lab pool')
    return keep, small


# ================================================== 1. locate the datasets ==
print('=== 1. Datasets ===')
disk_free('start')
SEARCH = [INPUT, WORK]
load_fp_cache()


def locate(explicit, splits, marker):
    if explicit:
        if not os.path.isdir(explicit):
            raise RuntimeError(f'{explicit} does not exist')
        return explicit
    return find_split_root(SEARCH, splits, marker)


t0 = time.time()
pv_root = locate(PV_ROOT, ('train', 'valid'), 'Tomato___healthy')
if pv_root is None:
    sh(f'kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p {WORK}/data --unzip')
    pv_root = find_split_root([f'{WORK}/data'], ('train', 'valid'), 'Tomato___healthy')
if pv_root is None:
    raise RuntimeError('PlantVillage train/valid not found - mount it as a Kaggle Input.')
PV_TRAIN, PV_VALID = os.path.join(pv_root, 'train'), os.path.join(pv_root, 'valid')

pd_root = locate(PD_ROOT, ('train', 'test'), 'Tomato leaf')
if pd_root is None:
    print('PlantDoc not found in inputs -> cloning (put it in pipeline_cache or set PD_ROOT to skip)')
    sh(f'git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git {WORK}/plantdoc')
    pd_root = f'{WORK}/plantdoc'
PD_TRAIN, PD_TEST = os.path.join(pd_root, 'train'), os.path.join(pd_root, 'test')
if not os.path.isdir(PD_TEST):
    raise RuntimeError('PlantDoc unavailable - enable internet or mount it as a Kaggle Input.')

extra_root = locate(EXTRA_ROOT, ('train',), 'Tomato_mosaic_virus')
print(f'PlantVillage: {pv_root}\nPlantDoc    : {pd_root}\nTomato extra: {extra_root or "not mounted"}')
print(f'Precision   : {PRECISION} | discovery took {time.time() - t0:.0f}s')

class_names = [c for c in subdirs(PV_TRAIN) if CROP is None or c.startswith(f'{CROP}___')]
NUM_CLASSES = len(class_names)
CLASS_INDEX = {c: i for i, c in enumerate(class_names)}
SHORT = [c.split('___', 1)[1] for c in class_names]
assert NUM_CLASSES > 1, f'no classes found for crop {CROP!r}'
print(f'{NUM_CLASSES} classes')
with open(f'{WORK}/class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)


# ============================================== 2. build the file indexes ==
print('\n=== 2. Index images ===')
identity = {c: c for c in class_names}
lab_items = collect(PV_TRAIN, identity, cap=None, tag='plantvillage')
pv_valid_items = collect(PV_VALID, identity, cap=None, tag='plantvillage-valid')

print('Fingerprinting the whole PlantDoc test split (all crops, all flips/rotations)...')
test_ref = reference([os.path.join(PD_TEST, d, f) for d in subdirs(PD_TEST)
                      for f in image_files(os.path.join(PD_TEST, d))])
test_items = collect(PD_TEST, build_mapping(PD_TEST, class_names, PLANTDOC_ALIASES, 'plantdoc-test'),
                     cap=None, tag='plantdoc-test')

pd_items = collect(PD_TRAIN, build_mapping(PD_TRAIN, class_names, PLANTDOC_ALIASES, 'plantdoc'),
                   cap=None, tag='plantdoc-train')
pd_items = drop_matches(pd_items, test_ref, 'plantdoc-train vs test')
val_cut = int(round(PD_VAL_FRACTION * 100))
val_items = [it for it in pd_items if stable_bucket(os.path.relpath(it[0], PD_TRAIN)) < val_cut]
field_items = [it for it in pd_items if stable_bucket(os.path.relpath(it[0], PD_TRAIN)) >= val_cut]
val_ref = reference([p for p, _ in val_items])
field_items = drop_matches(field_items, val_ref, 'plantdoc-train vs val')
holdout_ref = merge_refs(test_ref, val_ref)

print('Checking PlantVillage against the PlantDoc holdout (first run is slow, then cached)...')
lab_items = drop_matches(lab_items, holdout_ref, 'plantvillage vs holdout')

wild_items = []
for i, extra in enumerate(EXTRA_FIELD_DIRS):
    items = collect(extra, build_mapping(extra, class_names, tag=f'field-extra{i}'), tag=f'field-extra{i}')
    wild_items += drop_matches(items, holdout_ref, f'field-extra{i} vs holdout')

if extra_root and CROP in (None, 'Tomato'):
    items = drop_matches(collect_extra(extra_root), holdout_ref, 'tomato-extra vs holdout')
    items = drop_matches(items, reference([p for p, _ in field_items]), 'tomato-extra dup of plantdoc-train')
    if EXTRA_WILD_TO_FIELD:
        wild_items += items
    else:
        lab_items += items

save_fp_cache()
pd.DataFrame(removed_rows, columns=['source', 'path', 'reason']).to_csv(
    f'{WORK}/removed_images.csv', index=False)
if not test_items:
    raise RuntimeError('PlantDoc test is empty after mapping - check mapping_plantdoc-test.csv')

pd_pool, small_a = split_small(field_items, 'plantdoc-train')
wild_pool, small_b = split_small(wild_items, 'wild')
lab_items += small_a + small_b


def field_count(i):
    return len(pd_pool.get(i, [])) + len(wild_pool.get(i, []))


test_support = np.bincount([i for _, i in test_items], minlength=NUM_CLASSES)
val_support = np.bincount([i for _, i in val_items], minlength=NUM_CLASSES)
supported_idx = [i for i in range(NUM_CLASSES) if test_support[i] > 0]
val_scored_idx = [i for i in range(NUM_CLASSES) if val_support[i] > 0]

n_pd, n_wild = sum(map(len, pd_pool.values())), sum(map(len, wild_pool.values()))
print(f'\nLab: {len(lab_items)} | Field: PlantDoc {n_pd} + wild {n_wild}')
print(f'PlantDoc val: {len(val_items)} | test: {len(test_items)} ({len(supported_idx)} scorable classes)')
if len(val_items) < 100:
    print('NOTE: val has <100 images - one image = ~1 point; treat small val changes as noise.')
pd.DataFrame({'Class': class_names,
              'Lab images': np.bincount([i for _, i in lab_items], minlength=NUM_CLASSES),
              'PlantDoc field': [len(pd_pool.get(i, [])) for i in range(NUM_CLASSES)],
              'Wild field': [len(wild_pool.get(i, [])) for i in range(NUM_CLASSES)],
              'PlantDoc val support': val_support, 'PlantDoc test support': test_support}
             ).to_csv(f'{WORK}/class_coverage.csv', index=False)


# ================================================== 3. tf.data input pipe ==
def _resize(img, size):
    return tf.image.resize(img, size, antialias=True)   # antialias = matches PIL / serving


def _decode(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    return tf.image.convert_image_dtype(img, tf.float32)


def _center_square(img):
    s = tf.minimum(tf.shape(img)[0], tf.shape(img)[1])
    return _resize(tf.image.resize_with_crop_or_pad(img, s, s), (IMG_SIZE, IMG_SIZE))


def _random_resized_crop(img, min_scale=0.35):
    shape = tf.cast(tf.shape(img)[:2], tf.float32)
    area = shape[0] * shape[1] * tf.random.uniform([], min_scale, 1.0)
    ratio = tf.exp(tf.random.uniform([], tf.math.log(0.75), tf.math.log(1.33)))
    ch = tf.cast(tf.minimum(tf.sqrt(area / ratio), shape[0]), tf.int32)
    cw = tf.cast(tf.minimum(tf.sqrt(area * ratio), shape[1]), tf.int32)
    y = tf.random.uniform([], 0, tf.shape(img)[0] - ch + 1, tf.int32)
    x = tf.random.uniform([], 0, tf.shape(img)[1] - cw + 1, tf.int32)
    return _resize(tf.image.crop_to_bounding_box(img, y, x, ch, cw), (IMG_SIZE, IMG_SIZE))


def _sometimes(p, fn, img):
    return tf.cond(tf.random.uniform([]) < p, lambda: fn(img), lambda: img)


def _soften(img):
    small = tf.cast(IMG_SIZE * tf.random.uniform([], 0.3, 0.7), tf.int32)
    return tf.image.resize(_resize(img, (small, small)), (IMG_SIZE, IMG_SIZE))


def _noise(img):
    return img + tf.random.normal(tf.shape(img), stddev=tf.random.uniform([], 0.01, 0.05))


def _jpeg(img):
    return tf.image.random_jpeg_quality(img, 35, 95)


def _box_mask(h, w):
    y = tf.random.uniform([], 0, IMG_SIZE - h + 1, tf.int32)
    x = tf.random.uniform([], 0, IMG_SIZE - w + 1, tf.int32)
    rows, cols = tf.range(IMG_SIZE)[:, None], tf.range(IMG_SIZE)[None, :]
    return tf.cast((rows >= y) & (rows < y + h) & (cols >= x) & (cols < x + w), tf.float32)


def _erase(img):
    h = tf.random.uniform([], IMG_SIZE // 10, IMG_SIZE // 3, tf.int32)
    w = tf.random.uniform([], IMG_SIZE // 10, IMG_SIZE // 3, tf.int32)
    m = _box_mask(h, w)[..., None]
    return img * (1 - m) + tf.random.uniform(tf.shape(img)) * m


def _augment(img):
    img = _random_resized_crop(img)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, tf.random.uniform([], 0, 4, tf.int32))
    img = tf.image.random_brightness(img, 0.25)
    img = tf.image.random_contrast(img, 0.7, 1.4)
    img = tf.clip_by_value(img, 0.0, 1.0)
    img = tf.image.random_saturation(img, 0.6, 1.5)
    img = tf.image.random_hue(img, 0.04)
    img = tf.clip_by_value(img, 0.0, 1.0)
    img = _sometimes(0.30, _jpeg, img)
    img = _sometimes(0.25, _soften, img)
    img = _sometimes(0.25, _noise, img)
    img = _sometimes(0.30, _erase, img)
    return tf.ensure_shape(tf.clip_by_value(img, 0.0, 1.0), [IMG_SIZE, IMG_SIZE, 3])


def _load_train(path, label):
    return _augment(_decode(path)) * 255.0, tf.one_hot(label, NUM_CLASSES)   # model contract: 0..255


def _load_eval(path, label):
    return _center_square(_decode(path)) * 255.0, tf.one_hot(label, NUM_CLASSES)


def _ignore_errors(ds):
    return ds.ignore_errors() if hasattr(ds, 'ignore_errors') \
        else ds.apply(tf.data.experimental.ignore_errors())


def _paths_ds(items, shuffle, seed=SEED):
    ds = tf.data.Dataset.from_tensor_slices((tf.constant([p for p, _ in items], tf.string),
                                             tf.constant([l for _, l in items], tf.int32)))
    if shuffle:
        ds = ds.shuffle(len(items), seed=seed, reshuffle_each_iteration=True).repeat()
    return ds


def _mixture(parts, weights, seed):
    if len(parts) == 1:
        return parts[0]
    w = np.asarray(weights, float)
    return tf.data.Dataset.sample_from_datasets(parts, (w / w.sum()).tolist(), seed=seed)


def _balanced(pool, seed):
    parts, weights = [], []
    for k, (idx, paths) in enumerate(sorted(pool.items())):
        parts.append(_paths_ds([(p, idx) for p in paths], True, seed + k))
        weights.append(len(paths) ** FIELD_BALANCE_TEMP)
    return _mixture(parts, weights, seed)


def _beta(alpha):
    g1, g2 = tf.random.gamma([], alpha), tf.random.gamma([], alpha)
    return g1 / tf.maximum(g1 + g2, 1e-7)


def _mixup(x, y, idx):
    lam = _beta(MIXUP_ALPHA)
    return lam * x + (1 - lam) * tf.gather(x, idx), lam * y + (1 - lam) * tf.gather(y, idx)


def _cutmix(x, y, idx):
    side = tf.cast(IMG_SIZE * tf.sqrt(1.0 - _beta(CUTMIX_ALPHA)), tf.int32)
    m = _box_mask(side, side)
    lam = 1.0 - tf.reduce_mean(m)
    m = m[None, ..., None]
    return x * (1 - m) + tf.gather(x, idx) * m, lam * y + (1 - lam) * tf.gather(y, idx)


def _mix_batch(x, y):
    idx = tf.random.shuffle(tf.range(tf.shape(x)[0]))
    return tf.cond(tf.random.uniform([]) < CUTMIX_PROB,
                   lambda: _cutmix(x, y, idx), lambda: _mixup(x, y, idx))


def train_dataset(field_mix, mix, seed):
    field, fw = [], []
    if pd_pool:
        field.append(_balanced(pd_pool, seed)); fw.append(PLANTDOC_SHARE)
    if wild_pool:
        field.append(_balanced(wild_pool, seed + 100)); fw.append(1.0 - PLANTDOC_SHARE)
    parts, weights = [], []
    if field:
        parts.append(_mixture(field, fw, seed + 200)); weights.append(field_mix)
    if lab_items:
        parts.append(_paths_ds(lab_items, True, seed + 300)); weights.append(1.0 - field_mix)
    if not parts:
        raise RuntimeError('No training data found.')
    ds = _mixture(parts, weights, seed + 400)
    ds = _ignore_errors(ds.map(_load_train, num_parallel_calls=AUTOTUNE))
    ds = ds.batch(BATCH, drop_remainder=True)
    if mix:
        ds = ds.map(_mix_batch, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


def eval_dataset(items, cache=False):
    ds = _ignore_errors(_paths_ds(items, shuffle=False).map(_load_eval, num_parallel_calls=AUTOTUNE))
    if cache:
        ds = ds.cache()
    return ds.batch(BATCH).prefetch(AUTOTUNE)


pd_val_ds = eval_dataset(val_items, cache=True)
pd_test_ds = eval_dataset(test_items)
pv_valid_ds = eval_dataset(pv_valid_items)


# -------------------------------------------- inference (no model.predict) --
def _zoom(x, f):
    c = int(IMG_SIZE * f); o = (IMG_SIZE - c) // 2
    return _resize(tf.image.crop_to_bounding_box(x, o, o, c, c), (IMG_SIZE, IMG_SIZE))


def view_identity(x): return x
def view_hflip(x): return tf.image.flip_left_right(x)
def view_vflip(x): return tf.image.flip_up_down(x)
def view_rot90(x): return tf.image.rot90(x, 1)
def view_zoom85(x): return _zoom(x, 0.85)


TTA_VIEWS = (view_identity, view_hflip, view_vflip, view_rot90, view_zoom85)
_FORWARD = {}


def _forward(m):
    if id(m) not in _FORWARD:
        @tf.function(reduce_retracing=True)
        def fwd(x):
            return m(x, training=False)
        _FORWARD[id(m)] = fwd
    return _FORWARD[id(m)]


def predict(m, ds, tta=False):
    """One pass over `ds` -> (plain_probs, tta_probs, labels). Labels come from the same pass,
    so predictions and labels are always aligned."""
    fwd, views = _forward(m), (TTA_VIEWS if tta else TTA_VIEWS[:1])
    plain, avg, labels = [], [], []
    for x, y in ds:
        outs = [fwd(v(x)).numpy() for v in views]
        plain.append(outs[0]); avg.append(np.mean(outs, axis=0))
        labels.append(np.argmax(y.numpy(), axis=1))
    if not labels:
        return np.zeros((0, NUM_CLASSES)), np.zeros((0, NUM_CLASSES)), np.array([], int)
    return np.concatenate(plain), np.concatenate(avg), np.concatenate(labels)


# ======================================================= 4. model + train ==
def build_model():
    base = BACKBONES[BACKBONE](include_top=False, weights='imagenet',
                               input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)                   # BN always in inference mode
    x = keras.layers.GlobalAveragePooling2D(name='gap')(x)
    x = keras.layers.Dropout(DROPOUT, name='dropout')(x)
    outputs = keras.layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32', name='head')(x)
    return keras.Model(inputs, outputs, name='tomato_model'), base


def set_finetune(base):
    base.trainable = True
    for layer in base.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False
    n = sum(int(l.trainable) for l in base.layers)
    print(f'  fine-tune: {n}/{len(base.layers)} backbone layers trainable (BN frozen)')


def macro_f1(y, y_hat, labels):
    return float(f1_score(y, y_hat, labels=labels, average='macro', zero_division=0)) if len(y) else 0.0


def nll(p, y):
    return float(-np.log(p[np.arange(len(y)), y] + 1e-12).mean())


class FieldSelector(keras.callbacks.Callback):
    """Shadow-EMA of weights + early stopping on PlantDoc-val macro-F1 (ties -> lower NLL).
    Restores the best (EMA) weights at the end of the phase."""

    def __init__(self, patience, use_ema):
        super().__init__()
        self.patience, self.use_ema = patience, use_ema
        self.history = defaultdict(list)

    def on_train_begin(self, logs=None):
        self.vars = self.model.trainable_weights
        self.ema = [v.numpy().copy() for v in self.vars] if self.use_ema else None
        self.best, self.best_loss, self.best_w, self.wait, self.step = -1.0, np.inf, None, 0, 0

    def on_train_batch_end(self, batch, logs=None):
        if self.ema is None:
            return
        self.step += 1
        if self.step % EMA_EVERY == 0:
            d = EMA_DECAY ** EMA_EVERY
            for e, v in zip(self.ema, self.vars):
                e *= d
                e += (1.0 - d) * v.numpy()

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        raw = None
        if self.ema is not None:
            raw = [v.numpy() for v in self.vars]
            for v, e in zip(self.vars, self.ema):
                v.assign(e)
        probs, _, y = predict(self.model, pd_val_ds)
        y_hat = probs.argmax(1)
        f1, acc, loss = macro_f1(y, y_hat, val_scored_idx), float((y_hat == y).mean()), nll(probs, y)
        if f1 > self.best + 1e-4 or (abs(f1 - self.best) <= 1e-4 and loss < self.best_loss):
            self.best, self.best_loss, self.wait = f1, loss, 0
            self.best_w = [v.numpy().copy() for v in self.vars]
            mark = ' *'
        else:
            self.wait += 1
            mark = ''
            if self.wait >= self.patience:
                self.model.stop_training = True
        if raw is not None:
            for v, r in zip(self.vars, raw):
                v.assign(r)
        for k, val in [('loss', logs.get('loss')), ('acc', logs.get('acc')),
                       ('val_acc', acc), ('val_macro_f1', f1), ('val_loss', loss)]:
            if val is not None:
                self.history[k].append(float(val))
        print(f'  epoch {epoch + 1}: val_acc={acc:.3f} val_macro_f1={f1:.3f} val_loss={loss:.3f}{mark}',
              flush=True)

    def on_train_end(self, logs=None):
        if self.best_w is not None:
            for v, w in zip(self.vars, self.best_w):
                v.assign(w)
            print(f'  restored best weights (val_macro_f1={self.best:.3f})')


class Heartbeat(keras.callbacks.Callback):
    def __init__(self, every=100):
        super().__init__()
        self.every, self.t0 = every, time.time()

    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.time()

    def on_train_batch_end(self, batch, logs=None):
        if (batch + 1) % self.every == 0:
            logs = logs or {}
            print(f'    batch {batch + 1} loss={float(logs.get("loss", 0)):.4f} '
                  f'acc={float(logs.get("acc", 0)):.3f} ({time.time() - self.t0:.0f}s)', flush=True)


def run_phase(model, base, cfg, seed):
    print(f"\n--- Phase {cfg['name']}: {cfg['epochs']}x{cfg['steps']} steps, lr={cfg['lr']}, "
          f"field_mix={cfg['field_mix']:.0%}, mix={cfg['mix']}, ema={cfg['ema']} ---")
    if cfg['finetune']:
        set_finetune(base)
    total = cfg['epochs'] * cfg['steps']
    warm = max(1, cfg['steps'] // 2)
    schedule = keras.optimizers.schedules.CosineDecay(
        cfg['lr'] * 0.1, decay_steps=max(1, total - warm), alpha=0.05,
        warmup_target=cfg['lr'], warmup_steps=warm)
    model.compile(optimizer=keras.optimizers.AdamW(schedule, weight_decay=WEIGHT_DECAY, clipnorm=1.0),
                  loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
                  metrics=[keras.metrics.CategoricalAccuracy(name='acc')])
    selector = FieldSelector(cfg['patience'], cfg['ema'])
    model.fit(train_dataset(cfg['field_mix'], cfg['mix'], seed), epochs=cfg['epochs'],
              steps_per_epoch=cfg['steps'], verbose=2,
              callbacks=[selector, Heartbeat(max(50, cfg['steps'] // 4))])
    model.save_weights(CKPT_PATH)          # cheap per-phase checkpoint (no optimizer state)
    return dict(selector.history)


def export_inference_model(trained):
    """float32, optimizer-free copy for serving (~4x smaller, fast on CPU)."""
    trained.save_weights(CKPT_PATH)
    keras.mixed_precision.set_global_policy('float32')
    clean, _ = build_model()
    clean.load_weights(CKPT_PATH)
    clean.save(MODEL_PATH)
    os.remove(CKPT_PATH)
    return clean


print('\n=== 3. Train ===')
print(f'Backbone {BACKBONE} @ {IMG_SIZE}px, batch {BATCH}, {PRECISION}')
model, backbone = build_model()
history = {}
for k, cfg in enumerate(PHASES):
    history[cfg['name']] = run_phase(model, backbone, cfg, SEED + 1000 * k)
    with open(f'{WORK}/training_history.json', 'w') as f:
        json.dump(history, f, indent=2)
serve_model = export_inference_model(model)
print(f'\nSaved inference model -> {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1e6:.0f} MB)')


# ============================================================ 5. evaluate ==
def apply_temperature(p, t):
    z = np.log(p + 1e-12) / t
    z = np.exp(z - z.max(1, keepdims=True))
    return z / z.sum(1, keepdims=True)


def fit_temperature(p, y):
    grid = np.linspace(0.3, 3.0, 55)
    return float(grid[int(np.argmin([nll(apply_temperature(p, t), y) for t in grid]))])


def calibrated_nll(p, y):
    return nll(apply_temperature(p, fit_temperature(p, y)), y)


def gating_table(p, y, thresholds):
    conf, pred = p.max(1), p.argmax(1)
    return pd.DataFrame([{'min_confidence': round(float(t), 2), 'coverage': float((conf >= t).mean()),
                          'accuracy_when_answering': float((pred[conf >= t] == y[conf >= t]).mean())
                          if (conf >= t).any() else np.nan} for t in thresholds])


def bootstrap_ci(y, y_hat, labels, n=2000):
    rng = np.random.default_rng(SEED)
    accs, f1s = [], []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        accs.append((y[i] == y_hat[i]).mean())
        f1s.append(macro_f1(y[i], y_hat[i], labels))
    return np.percentile(accs, [2.5, 97.5]).tolist(), np.percentile(f1s, [2.5, 97.5]).tolist()


print('\n=== 4. Evaluate (exported float32 model = what you serve) ===')
val_plain, val_tta, y_val = predict(serve_model, pd_val_ds, tta=True)
test_plain, test_tta, y_test = predict(serve_model, pd_test_ds, tta=True)

# every decision below is made on VAL; test is only reported
nll_plain, nll_tta = calibrated_nll(val_plain, y_val), calibrated_nll(val_tta, y_val)
USE_TTA = nll_tta < nll_plain
print(f'val calibrated NLL plain {nll_plain:.3f} / TTA {nll_tta:.3f} -> use_tta={USE_TTA}')
val_p, test_p = (val_tta, test_tta) if USE_TTA else (val_plain, test_plain)
TEMPERATURE = fit_temperature(val_p, y_val)
val_c, test_c = apply_temperature(val_p, TEMPERATURE), apply_temperature(test_p, TEMPERATURE)

grid = np.round(np.arange(0.0, 1.0, 0.05), 2)
gate_val = gating_table(val_c, y_val, grid)
ok = gate_val[(gate_val.accuracy_when_answering >= GATE_TARGET_ACC) & (gate_val.coverage >= 0.3)]
THRESHOLD = float(ok.min_confidence.min()) if len(ok) else 0.5

y_hat = test_c.argmax(1)
acc = float((y_hat == y_test).mean())
f1m = macro_f1(y_test, y_hat, supported_idx)
acc_ci, f1_ci = bootstrap_ci(y_test, y_hat, supported_idx)
acc_plain = float((test_plain.argmax(1) == y_test).mean())
acc_tta = float((test_tta.argmax(1) == y_test).mean())
pv_probs, _, y_pv = predict(serve_model, pv_valid_ds)
pv_acc = float((pv_probs.argmax(1) == y_pv).mean())

print(f'PlantVillage valid (lab, optimistic: augmented copies) : {pv_acc * 100:.1f}%')
print(f'PlantDoc test, plain / TTA           : {acc_plain * 100:.1f}% / {acc_tta * 100:.1f}%')
print(f'PlantDoc test, chosen (TTA={USE_TTA})  : {acc * 100:.1f}% acc '
      f'[95% CI {acc_ci[0] * 100:.0f}-{acc_ci[1] * 100:.0f}] | macro-F1 {f1m:.3f} '
      f'[{f1_ci[0]:.2f}-{f1_ci[1]:.2f}]  (n={len(y_test)})')
print(f'Temperature {TEMPERATURE:.2f} | threshold {THRESHOLD:.2f} (picked on val)')

gate_test = gating_table(test_c, y_test, sorted({0.0, 0.3, 0.5, 0.7, 0.9, THRESHOLD}))
print('\nConfidence gating on test (calibrated):\n' + gate_test.to_string(index=False))
gate_val.to_csv(f'{WORK}/confidence_gating_val.csv', index=False)
gate_test.to_csv(f'{WORK}/confidence_gating_test.csv', index=False)

prec, rec, f1c, _ = precision_recall_fscore_support(y_test, y_hat, labels=list(range(NUM_CLASSES)),
                                                    zero_division=0)
per_class = pd.DataFrame({'Class': class_names, 'PlantDoc support': test_support,
                          'Field images': [field_count(i) for i in range(NUM_CLASSES)],
                          'Precision': prec.round(3), 'Recall': rec.round(3), 'F1': f1c.round(3)})
per_class.to_csv(f'{WORK}/per_class_metrics.csv', index=False)
measurable = per_class[per_class['PlantDoc support'] > 0].sort_values('F1', ascending=False)
print('\n' + measurable.to_string(index=False))

with open(f'{WORK}/evaluation_report.json', 'w') as f:
    json.dump({'backbone': BACKBONE, 'img_size': IMG_SIZE, 'num_classes': NUM_CLASSES,
               'scorable_classes': len(supported_idx), 'pv_valid_acc': pv_acc,
               'plantdoc': {'acc': acc, 'acc_ci95': acc_ci, 'macro_f1': f1m, 'macro_f1_ci95': f1_ci,
                            'acc_plain': acc_plain, 'acc_tta': acc_tta, 'n': int(len(y_test))},
               'use_tta': bool(USE_TTA), 'temperature': TEMPERATURE, 'threshold': THRESHOLD,
               'train_images': {'lab': len(lab_items), 'plantdoc_field': n_pd, 'wild_field': n_wild,
                                'val': len(val_items)},
               'removed_images': len(removed_rows),
               'per_class': per_class.to_dict('records')}, f, indent=2, default=float)
with open(f'{WORK}/serving_config.json', 'w') as f:
    json.dump({'img_size': IMG_SIZE, 'temperature': TEMPERATURE, 'threshold': THRESHOLD,
               'use_tta': bool(USE_TTA)}, f, indent=2)

# ------------------------------------------------------------------ plots ---
cm = confusion_matrix(y_test, y_hat, labels=range(NUM_CLASSES))
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=SHORT, yticklabels=SHORT)
plt.xlabel('predicted'); plt.ylabel('true'); plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8); plt.title(f'PlantDoc test confusion (n={len(y_test)})')
plt.tight_layout(); plt.savefig(f'{WORK}/confusion_matrix.png', dpi=200); plt.close()

plt.figure(figsize=(8, 6))
plt.barh(np.arange(len(measurable)), measurable['F1'], color='#4C72B0')
plt.yticks(np.arange(len(measurable)), [c.split('___', 1)[1] for c in measurable['Class']], fontsize=8)
plt.gca().invert_yaxis(); plt.xlabel('F1'); plt.tight_layout()
plt.savefig(f'{WORK}/per_class_f1.png', dpi=200); plt.close()

curve = [v for ph in history.values() for v in ph.get('val_macro_f1', [])]
plt.figure(figsize=(8, 4)); plt.plot(curve, marker='o'); plt.xlabel('epoch (all phases)')
plt.ylabel('val macro-F1'); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(f'{WORK}/val_curve.png', dpi=200); plt.close()

plt.figure(figsize=(6, 5))
bars = plt.bar(['PlantVillage (lab)', 'PlantDoc (field)'], [pv_acc * 100, acc * 100], 0.5, color='#4C72B0')
plt.errorbar([1], [acc * 100], yerr=[[(acc - acc_ci[0]) * 100], [(acc_ci[1] - acc) * 100]],
             fmt='none', ecolor='k', capsize=6)
for b, top in zip(bars, [pv_acc * 100, acc_ci[1] * 100]):      # label above the CI, not on it
    plt.text(b.get_x() + b.get_width() / 2, top + 1.5, f'{b.get_height():.1f}%', ha='center')
plt.ylim(0, 112); plt.ylabel('Accuracy %'); plt.title('Domain gap (95% CI on field)')
plt.tight_layout(); plt.savefig(f'{WORK}/domain_gap.png', dpi=200); plt.close()


# ====================================================== 6. package output ==
SERVE_PY = r'''# predict_tomato.py - preprocessing identical to training/evaluation.
# Flask: from predict_tomato import predict ; result = predict(request.files['image'].stream)
import json, os
import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps

HERE = os.path.dirname(os.path.abspath(__file__))
MODEL = tf.keras.models.load_model(os.path.join(HERE, 'model_tomato.keras'), compile=False)
with open(os.path.join(HERE, 'class_names.json')) as f:
    CLASSES = json.load(f)
with open(os.path.join(HERE, 'serving_config.json')) as f:
    CFG = json.load(f)
SIZE = CFG['img_size']


def _resize(x, size):
    return tf.image.resize(x, size, antialias=True)


def _prep(src):
    img = ImageOps.exif_transpose(Image.open(src)).convert('RGB')   # honour phone rotation
    x = tf.convert_to_tensor(np.asarray(img), tf.float32) / 255.0
    s = tf.minimum(tf.shape(x)[0], tf.shape(x)[1])
    x = _resize(tf.image.resize_with_crop_or_pad(x, s, s), (SIZE, SIZE))
    return x[None] * 255.0                                          # model expects 0..255, NOT /255


def view_identity(x): return x
def view_hflip(x): return tf.image.flip_left_right(x)
def view_vflip(x): return tf.image.flip_up_down(x)
def view_rot90(x): return tf.image.rot90(x, 1)
def view_zoom85(x):
    c = int(SIZE * 0.85); o = (SIZE - c) // 2
    return _resize(tf.image.crop_to_bounding_box(x, o, o, c, c), (SIZE, SIZE))


VIEWS = [view_identity, view_hflip, view_vflip, view_rot90, view_zoom85] if CFG['use_tta'] else [view_identity]


def predict(src, top_k=3):
    x = _prep(src)
    p = np.mean([MODEL(v(x), training=False).numpy()[0] for v in VIEWS], axis=0)
    z = np.log(p + 1e-12) / CFG['temperature']
    p = np.exp(z - z.max()); p /= p.sum()
    top = [(CLASSES[i], float(p[i])) for i in np.argsort(p)[::-1][:top_k]]
    label = top[0][0] if top[0][1] >= CFG['threshold'] else 'uncertain - retake photo closer, in good light'
    return {'label': label, 'confidence': top[0][1], 'top': top}
'''
with open(f'{WORK}/predict_tomato.py', 'w') as f:
    f.write(SERVE_PY)

print('\n=== 5. Package artifacts ===')
KEEP = {os.path.basename(MODEL_PATH), 'class_names.json', 'serving_config.json', 'predict_tomato.py',
        'evaluation_report.json', 'per_class_metrics.csv', 'class_coverage.csv',
        'confidence_gating_val.csv', 'confidence_gating_test.csv', 'removed_images.csv',
        'training_history.json', 'confusion_matrix.png', 'per_class_f1.png',
        'domain_gap.png', 'val_curve.png'}
for name in ('plantdoc', 'data'):
    shutil.rmtree(os.path.join(WORK, name), ignore_errors=True)
ART_ZIP = f'{WORK}/tomato_artifacts.zip'
with zipfile.ZipFile(ART_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for entry in sorted(os.listdir(WORK)):
        path = os.path.join(WORK, entry)
        if os.path.isfile(path) and (entry in KEEP or entry.startswith('mapping_')):
            zf.write(path, entry, zipfile.ZIP_STORED if entry.endswith('.keras') else zipfile.ZIP_DEFLATED)
print(f'{ART_ZIP} ({os.path.getsize(ART_ZIP) / 1e6:.1f} MB)')
print(f'Add {WORK}/{FP_CACHE_NAME} to your pipeline_cache dataset to skip fingerprinting next run.')
disk_free('end')
print("\nServing: unzip next to your Flask app, then  from predict_tomato import predict")